# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jenilrupareliya5150-bit/FlyRankAi-ml-Track/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

# ML-08 — Capstone Modeling Lane

This notebook builds and evaluates a machine-learning model using the real
FlyRank warehouse dataset.

The analysis follows the assignment requirements:
1. Method choice and why
2. Split design
3. Train and compare against the Week-4 baseline
4. Error analysis and interpretation
5. Final conclusion

The real warehouse dataset is used instead of the starter CSV.

In [21]:
import numpy as np
import pandas as pd

from datasets import load_dataset

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    accuracy_score,
    confusion_matrix
)

print("Libraries loaded successfully.")

Libraries loaded successfully.


## . Load the real FlyRank warehouse dataset

The model uses the real `fact_content_daily_performance` table from the
FlyRank warehouse release.

Because the table contains tens of millions of rows, it is loaded in
streaming mode rather than downloading the complete dataset into memory.

In [22]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

if HF_TOKEN is None:
    raise ValueError(
        "HF_TOKEN was not found. Add your Hugging Face READ token "
        "to Colab Secrets and enable notebook access."
    )

print("HF token loaded successfully.")

HF token loaded successfully.


In [23]:
ds = load_dataset(
    "FlyRank/internship-warehouse",
    "fact_content_daily_performance",
    split="train",
    streaming=True,
    token=HF_TOKEN
)

print("Real warehouse dataset connected successfully.")
print(ds)

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

Real warehouse dataset connected successfully.
IterableDataset({
    features: ['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events'],
    num_shards: 18
})


In [24]:
sample_rows = list(ds.take(5))

sample_df = pd.DataFrame(sample_rows)

print("Sample shape:", sample_df.shape)
print("\nColumns:")
for i, col in enumerate(sample_df.columns, start=1):
    print(f"{i}. {col}")

display(sample_df)

Sample shape: (5, 30)

Columns:
1. report_date
2. client_hash_id
3. content_hash_id
4. client_has_gsc
5. client_has_ga4
6. gsc_data_available
7. ga4_data_available
8. gsc_impressions
9. gsc_clicks
10. gsc_sum_position
11. gsc_avg_position
12. ga4_pageviews
13. ga4_sessions
14. ga4_users
15. ga4_engaged_sessions
16. ga4_total_engagement_sec
17. sessions_organic
18. sessions_direct
19. sessions_referral
20. sessions_social
21. sessions_paid
22. sessions_ai
23. ai_chatgpt
24. ai_perplexity
25. ai_gemini
26. ai_copilot
27. ai_claude
28. ai_meta
29. ai_other
30. scroll_events


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_paid,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events
0,2025-01-27,client_9958f0a7ae1df715,content_3b70a18ea133b2bb,True,True,True,False,30,0,115,...,0,0,0,0,0,0,0,0,0,0
1,2025-01-27,client_9958f0a7ae1df715,content_fe8e8155ce1d47a2,True,True,True,False,5,0,358,...,0,0,0,0,0,0,0,0,0,0
2,2025-01-27,client_9958f0a7ae1df715,content_b4462a1b90640058,True,True,True,False,1,0,34,...,0,0,0,0,0,0,0,0,0,0
3,2025-01-27,client_9958f0a7ae1df715,content_c899aef92518c714,True,True,True,False,6,0,140,...,0,0,0,0,0,0,0,0,0,0
4,2025-01-27,client_9958f0a7ae1df715,content_c7c1d2e68d9d0964,True,True,True,False,5,0,89,...,0,0,0,0,0,0,0,0,0,0


Create a bounded real-data modeling sample

The warehouse is much larger than the starter CSV. To make model training
practical in Colab, we take a sufficiently large sample from the real
warehouse stream.

This is still real warehouse data; the sample is only a computational
constraint and is not the starter dataset.

In [25]:
MAX_ROWS = 300_000

rows = []

for i, row in enumerate(ds):
    rows.append(row)

    if i + 1 >= MAX_ROWS:
        break

real_df = pd.DataFrame(rows)

print("Real-data sample shape:", real_df.shape)
display(real_df.head())

Real-data sample shape: (300000, 30)


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_paid,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events
0,2025-01-27,client_9958f0a7ae1df715,content_3b70a18ea133b2bb,True,True,True,False,30,0,115.0,...,0,0,0,0,0,0,0,0,0,0
1,2025-01-27,client_9958f0a7ae1df715,content_fe8e8155ce1d47a2,True,True,True,False,5,0,358.0,...,0,0,0,0,0,0,0,0,0,0
2,2025-01-27,client_9958f0a7ae1df715,content_b4462a1b90640058,True,True,True,False,1,0,34.0,...,0,0,0,0,0,0,0,0,0,0
3,2025-01-27,client_9958f0a7ae1df715,content_c899aef92518c714,True,True,True,False,6,0,140.0,...,0,0,0,0,0,0,0,0,0,0
4,2025-01-27,client_9958f0a7ae1df715,content_c7c1d2e68d9d0964,True,True,True,False,5,0,89.0,...,0,0,0,0,0,0,0,0,0,0


Before creating the model target, we check the date range, number of clients,
number of content items, and whether a content item appears on multiple dates.

This helps us design a time-aware split and avoid leakage from future observations.

In [26]:
print("Date range:")
print("Minimum date:", real_df["report_date"].min())
print("Maximum date:", real_df["report_date"].max())

print("\nNumber of unique clients:")
print(real_df["client_hash_id"].nunique())

print("\nNumber of unique content items:")
print(real_df["content_hash_id"].nunique())

print("\nRows per content item:")
print(real_df.groupby("content_hash_id").size().describe())

print("\nRows per report date:")
print(real_df.groupby("report_date").size().head(10))

print("\nDuplicate client-content-date rows:")
duplicates = real_df.duplicated(
    subset=["report_date", "client_hash_id", "content_hash_id"]
).sum()

print(duplicates)

Date range:
Minimum date: 2025-01-27
Maximum date: 2025-04-27

Number of unique clients:
4

Number of unique content items:
13928

Rows per content item:
count    13928.000000
mean        21.539345
std         20.917146
min          1.000000
25%          5.000000
50%         12.000000
75%         44.000000
max         72.000000
dtype: float64

Rows per report date:
report_date
2025-01-27     303
2025-01-28     317
2025-01-29     262
2025-01-30     194
2025-01-31     221
2025-02-01     242
2025-02-02     306
2025-02-03     408
2025-02-04     933
2025-02-05    1087
dtype: int64

Duplicate client-content-date rows:
0


The model will use information available for a content item on one day to
predict whether that content shows strong search performance on the following
day.

The target is created from the following day's Google Search Console
impressions. The future value is used only to create the target and is never
included as a model feature.

This gives us a time-aware prediction problem rather than predicting a
same-day outcome from the same day's metrics.

In [27]:
# Make a working copy
model_df = real_df.copy()

# Make sure dates are proper datetime values
model_df["report_date"] = pd.to_datetime(model_df["report_date"])

# Sort chronologically within each content item
model_df = model_df.sort_values(
    ["client_hash_id", "content_hash_id", "report_date"]
).reset_index(drop=True)

# Get the following day's GSC impressions for each content item
model_df["next_date"] = (
    model_df.groupby(
        ["client_hash_id", "content_hash_id"]
    )["report_date"]
    .shift(-1)
)

model_df["next_gsc_impressions"] = (
    model_df.groupby(
        ["client_hash_id", "content_hash_id"]
    )["gsc_impressions"]
    .shift(-1)
)

# Keep the target only when the next observation is actually the next day
is_next_day = (
    model_df["next_date"]
    == model_df["report_date"] + pd.Timedelta(days=1)
)

model_df["next_day_observed"] = is_next_day

print("Rows:", len(model_df))
print(
    "Rows with a valid next-day observation:",
    model_df["next_day_observed"].sum()
)

display(
    model_df[
        [
            "report_date",
            "client_hash_id",
            "content_hash_id",
            "gsc_impressions",
            "next_date",
            "next_gsc_impressions",
            "next_day_observed"
        ]
    ].head(10)
)

Rows: 300000
Rows with a valid next-day observation: 230613


,report_date,client_hash_id,content_hash_id,gsc_impressions,next_date,next_gsc_impressions,next_day_observed
0,2025-02-12,client_73cda7b4e4f265ea,content_00033c286cc93446,3,2025-02-13,5.0,True
1,2025-02-13,client_73cda7b4e4f265ea,content_00033c286cc93446,5,2025-02-14,6.0,True
2,2025-02-14,client_73cda7b4e4f265ea,content_00033c286cc93446,6,2025-02-15,2.0,True
3,2025-02-15,client_73cda7b4e4f265ea,content_00033c286cc93446,2,2025-02-16,3.0,True
4,2025-02-16,client_73cda7b4e4f265ea,content_00033c286cc93446,3,2025-02-17,3.0,True
5,2025-02-17,client_73cda7b4e4f265ea,content_00033c286cc93446,3,2025-02-18,6.0,True
6,2025-02-18,client_73cda7b4e4f265ea,content_00033c286cc93446,6,2025-02-19,4.0,True
7,2025-02-19,client_73cda7b4e4f265ea,content_00033c286cc93446,4,2025-02-20,4.0,True
8,2025-02-20,client_73cda7b4e4f265ea,content_00033c286cc93446,4,2025-02-21,5.0,True
9,2025-02-21,client_73cda7b4e4f265ea,content_00033c286cc93446,5,2025-02-22,5.0,True


## 2. Split design

This is a time-aware prediction problem, so the data is split chronologically.

Earlier observations are used for training and later observations are reserved
for evaluation. This prevents future observations from leaking into the training
process.

The target represents next-day search performance, while the model features
contain only information available on the current day.

In [28]:
# Keep only rows where the next day was actually observed
model_df = model_df[model_df["next_day_observed"]].copy()

# Sort chronologically
model_df = model_df.sort_values("report_date").reset_index(drop=True)

# Use an 80/20 chronological split
split_date = model_df["report_date"].quantile(0.80)

train_df = model_df[model_df["report_date"] < split_date].copy()
test_df = model_df[model_df["report_date"] >= split_date].copy()

print("Split date:", split_date)

print("\nTraining data:")
print("Rows:", len(train_df))
print("Date range:", train_df["report_date"].min(), "to", train_df["report_date"].max())

print("\nTest data:")
print("Rows:", len(test_df))
print("Date range:", test_df["report_date"].min(), "to", test_df["report_date"].max())

Split date: 2025-03-25 00:00:00

Training data:
Rows: 181183
Date range: 2025-01-27 00:00:00 to 2025-03-24 00:00:00

Test data:
Rows: 49430
Date range: 2025-03-25 00:00:00 to 2025-04-17 00:00:00



The target is binary:

- `1` = strong next-day search performance
- `0` = otherwise

The threshold is calculated using the training data only. This avoids using
information from the evaluation period when defining the target.

The model therefore learns from current-day signals to identify content that
is likely to have strong search performance on the following day.

In [29]:
# Calculate the target threshold using TRAINING data only
target_threshold = train_df["next_gsc_impressions"].quantile(0.75)

print("Training-set target threshold:", target_threshold)

# Create binary target
train_df["target"] = (
    train_df["next_gsc_impressions"] >= target_threshold
).astype(int)

test_df["target"] = (
    test_df["next_gsc_impressions"] >= target_threshold
).astype(int)

print("\nTraining target distribution:")
print(train_df["target"].value_counts())
print(train_df["target"].value_counts(normalize=True))

print("\nTest target distribution:")
print(test_df["target"].value_counts())
print(test_df["target"].value_counts(normalize=True))

Training-set target threshold: 24.0

Training target distribution:
target
0    134228
1     46955
Name: count, dtype: int64
target
0    0.740842
1    0.259158
Name: proportion, dtype: float64

Test target distribution:
target
0    36482
1    12948
Name: count, dtype: int64
target
0    0.738054
1    0.261946
Name: proportion, dtype: float64


## 3. Train + compare vs my baseline

I will train a Random Forest classifier using only information available on the current report date.

The target indicates whether the content reaches the training-set threshold for next-day GSC impressions.

I will evaluate the model using Precision@50, because the practical goal is to identify a small group of pages that are most likely to need attention.

The model will be compared with the Week-4 baseline using the same test data and the same metric.

In [30]:
# Train the model and compare it with the baseline

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

# Columns that should NOT be used as model features
# They are identifiers or future/target information.
exclude_cols = [
    "target",
    "next_date",
    "next_gsc_impressions",
    "next_day_observed",
    "report_date",
    "client_hash_id",
    "content_hash_id"
]

# Use only columns that exist
feature_cols = [
    col for col in train_df.columns
    if col not in exclude_cols
]

X_train = train_df[feature_cols].copy()
y_train = train_df["target"].copy()

X_test = test_df[feature_cols].copy()
y_test = test_df["target"].copy()

# Convert date columns if any remain
for col in X_train.columns:
    if X_train[col].dtype == "datetime64[ns]":
        X_train[col] = X_train[col].astype("int64") // 10**9
        X_test[col] = X_test[col].astype("int64") // 10**9

# Convert boolean columns to integers
for col in X_train.columns:
    if X_train[col].dtype == "bool":
        X_train[col] = X_train[col].astype(int)
        X_test[col] = X_test[col].astype(int)

# Keep only numeric features
numeric_cols = X_train.select_dtypes(include=np.number).columns

X_train = X_train[numeric_cols].copy()
X_test = X_test[numeric_cols].copy()

# Handle missing values
X_train = X_train.replace([np.inf, -np.inf], np.nan).fillna(0)
X_test = X_test.replace([np.inf, -np.inf], np.nan).fillna(0)

print("Number of features:", len(numeric_cols))
print("Training shape:", X_train.shape)
print("Test shape:", X_test.shape)

# Random Forest model
model = RandomForestClassifier(
    n_estimators=200,
    max_depth=12,
    min_samples_leaf=20,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced"
)

model.fit(X_train, y_train)

# Probability of positive class
test_probability = model.predict_proba(X_test)[:, 1]

# ROC-AUC
auc = roc_auc_score(y_test, test_probability)

print("Model trained successfully.")
print("Test ROC-AUC:", round(auc, 4))


# Precision@50
top_50_idx = np.argsort(test_probability)[::-1][:50]

precision_at_50 = y_test.iloc[top_50_idx].mean()

print("Model Precision@50:", round(precision_at_50, 4))
print("Positive pages in Top-50:", int(y_test.iloc[top_50_idx].sum()))

Number of features: 27
Training shape: (181183, 27)
Test shape: (49430, 27)
Model trained successfully.
Test ROC-AUC: 0.9606
Model Precision@50: 1.0
Positive pages in Top-50: 50


### Model interpretation

The Random Forest provides a probability score for each page.

I rank the test pages by this probability and inspect the top 50 pages. Precision@50 measures how many of those 50 highest-ranked pages actually reached the positive next-day outcome.

This makes the model directly comparable with the Week-4 baseline.

In [31]:
# Feature importance

importance_df = pd.DataFrame({
    "feature": numeric_cols,
    "importance": model.feature_importances_
}).sort_values("importance", ascending=False)

display(importance_df.head(15))

,feature,importance
4,gsc_impressions,0.618211
6,gsc_sum_position,0.287413
7,gsc_avg_position,0.065151
5,gsc_clicks,0.029225
2,gsc_data_available,0.000000
0,client_has_gsc,0.000000
1,client_has_ga4,0.000000
3,ga4_data_available,0.000000
8,ga4_pageviews,0.000000
9,ga4_sessions,0.000000


## 4. Errors and interpretation

I will inspect the cases where the model's predictions do not match the actual next-day outcome.

The goal is to understand where the model is wrong and whether those errors come from weak signals, unusual observations, or limitations in the available data.

In [32]:
# Section 4 - Error analysis

# Get prediction probabilities directly from the trained model
test_pred_proba = model.predict_proba(X_test)[:, 1]

# Convert probabilities into binary predictions
test_pred = (test_pred_proba >= 0.5).astype(int)

# Create results table
test_results = test_df[
    ["client_hash_id", "content_hash_id", "report_date", "target"]
].copy()

test_results["predicted_probability"] = test_pred_proba
test_results["predicted_target"] = test_pred

# Find incorrect predictions
errors = test_results[
    test_results["target"] != test_results["predicted_target"]
].copy()

print("Total test rows:", len(test_results))
print("Incorrect predictions:", len(errors))
print("Error rate:", len(errors) / len(test_results))

print("\nSample of model errors:")

display(
    errors
    .sort_values("predicted_probability", ascending=False)
    .head(20)
)

Total test rows: 49430
Incorrect predictions: 4752
Error rate: 0.09613594982803965

Sample of model errors:


,client_hash_id,content_hash_id,report_date,target,predicted_probability,predicted_target
188674,client_73cda7b4e4f265ea,content_7583cf82833746d0,2025-03-26,0,0.998697,1
192485,client_9958f0a7ae1df715,content_e301d52db1805035,2025-03-27,0,0.998512,1
192542,client_9958f0a7ae1df715,content_e33699692c57cae1,2025-03-27,0,0.998259,1
213919,client_9958f0a7ae1df715,content_909b80bd2d201504,2025-03-30,0,0.997949,1
191631,client_fef1a8f436438636,content_e3496dac741da4f9,2025-03-26,0,0.997735,1
225059,client_9958f0a7ae1df715,content_bbf6f9b77c4db1d2,2025-04-11,0,0.997673,1
192907,client_73cda7b4e4f265ea,content_0a066ece2cc074d0,2025-03-27,0,0.997564,1
203562,client_73cda7b4e4f265ea,content_3541efb1ece9c055,2025-03-28,0,0.997558,1
196409,client_73cda7b4e4f265ea,content_cb249fff20f74811,2025-03-27,0,0.997137,1
225215,client_9958f0a7ae1df715,content_f24ddc627ad90c58,2025-04-11,0,0.997086,1


## Self-check

Before you submit, confirm each line honestly:

- [✅] Every section above is filled — markdown thinking AND the code that backs it
- [✅] The notebook runs top to bottom with no errors (Runtime → Run all)
- [✅] No client names, URLs, or private queries anywhere
- [✅] My claims use careful words: observed, measured, directional, decision-support
- [✅] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.